# Predicting VMAT gamma passing rates in multi-target SRS

**Task:** predict the **gamma passing rate (GPR)** of each VMAT arc of single-isocenter multi-target
stereotactic radiosurgery (SIMT-SRS) plans from

- **arc-specific plan complexity metrics (aPCMs)**, and
- the **daily Machine Performance Check (MPC)** of the linac that delivered the arc.

The GPR comes from Varian Portal Dosimetry (gamma analysis, measured vs predicted EPID image).

**Workflow:** download → explore → merge → clean → analyse → model → interpret.

The column dictionary is in the repository `README.md`. Keep it open while you work.

> The data are pseudonymised clinical data. Use them only for this course; do not redistribute them.

## 0. Setup

In [ ]:
%pip install -q pyzipper gdown

In [ ]:
import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyzipper

pd.set_option("display.max_columns", 60)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 1. Download and unlock the data

The instructor shares the Google Drive link and, during the lecture, the **passkey**.

In [ ]:
import gdown


def download_and_unlock(drive_url, zip_name):
    """Download an encrypted archive from Google Drive and extract it into DATA_DIR."""
    zip_path = Path(zip_name)
    if not zip_path.exists():
        gdown.download(drive_url, str(zip_path), fuzzy=True, quiet=False)
    passkey = getpass.getpass(f"Passkey for {zip_name}: ")
    with pyzipper.AESZipFile(zip_path) as z:
        z.setpassword(passkey.encode("utf-8"))
        z.extractall(DATA_DIR)
    print("Files in data/:", sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
DRIVE_URL = "https://drive.google.com/file/d/1m3jGyGtCycGGZM8XmzwPGaNVMbWIoN0R/view?usp=sharing"

download_and_unlock(DRIVE_URL, "srs_gpr_data.zip")

## 2. Load the raw tables

In [ ]:
raw = pd.read_excel(DATA_DIR / "srs_gpr_raw.xlsx", sheet_name=None)
for name, df in raw.items():
    print(f"{name:18} {df.shape}")

beams = raw["BeamInfoHyperArc"]
pd_info = raw["PDInfo"]
complexity = raw["BeamComplexity"]
mpc = raw["MPC"]

## 3. Explore each table

For each table answer:

- What does **one row** represent?
- Which columns are **keys** that link to other tables? Are they unique?
- Which columns are **constant** or almost empty (useless as features)?
- What are the **types** and **units**? Anything stored as text that should be a number or a date?
- Are there **duplicates**?

In [ ]:
beams.head()

In [ ]:
# TODO: explore beams, pd_info, complexity, mpc
# Hints: .info(), .describe(), .nunique(), .isna().mean(), .value_counts()

In [ ]:
# TODO: how many portal dosimetry analyses per arc? Which gamma criteria are present?

## 4. Merge the tables

Build **one row per arc** with: identifiers, plan info, aPCMs, the GPR, and the MPC results of the
delivery day.

Relationships (check them yourself; they are not always a clean 1-to-1 match):

| From | To | Keys |
|---|---|---|
| `BeamInfoHyperArc` | `BeamComplexity` | `PatientID`, `Plan`, `Beam` ↔ `Field` |
| `BeamInfoHyperArc` | `PDInfo` | `PatientID`, `PDPlan` ↔ `Plan`, `Beam` ↔ `PDBeam` |
| `PDInfo` | `MPC` | `MachineName` (from the beam table), **date** of `SessionDateTime` ↔ **date** of `Date` |

Before and after every merge, check the number of rows. Did it grow (duplicated keys)? Did it shrink
(keys that do not match)? `pd.merge(..., validate=..., indicator=True)` helps.

In [ ]:
# TODO: beams + complexity

In [ ]:
# TODO: + portal dosimetry (choose the right gamma criterion; handle repeated analyses)

In [ ]:
# TODO: + MPC (parse dates, aggregate the runs of each day per machine, then merge on machine + date)

## 5. Clean

Checklist:

- [ ] Keep only the relevant plans (multi-target SRS, `SRS ARC`, HyperArc).
- [ ] Keep only one gamma criterion: **3% / 1 mm, 20% threshold**.
- [ ] Decide what to do with repeated measurements of the same arc.
- [ ] Use the same unit for GPR everywhere (fraction or %).
- [ ] Convert text columns to numbers or dates (`DosePerFraction`, MPC `Date`, …).
- [ ] Remove constant and duplicated columns.
- [ ] Handle missing values (e.g. days without MPC).
- [ ] Look for outliers in the GPR and in the features.

In [ ]:
# TODO: build `df_model`, one row per arc, ready for modelling
df_model = None

## 6. Exploratory analysis

- What does the GPR distribution look like? Is it skewed? How many arcs fall below a clinical action level?
- Which aPCMs correlate with the GPR? Which aPCMs are **strongly correlated with each other**?
- How do the MPC results change over time and between the two linacs?
- Does the GPR depend on the machine, the energy, or the number of lesions?

In [ ]:
# TODO: plots

## 7. Modelling

**Important:** each patient has several arcs, and arcs of the same patient are similar. Split the data
**by patient** (`GroupKFold`) so the test set contains only patients the model has never seen.

Start from the baseline below, then try other models and feature sets:

1. aPCMs only
2. aPCMs + MPC
3. Feature selection or dimensionality reduction (many aPCMs are redundant)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupKFold, cross_validate


def evaluate(model, X, y, groups, n_splits=5):
    # Patient-grouped cross-validation: returns mean ± std of MAE, RMSE and R²
    cv = GroupKFold(n_splits=n_splits)
    scores = cross_validate(
        model, X, y, groups=groups, cv=cv,
        scoring={"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"},
    )
    out = {}
    for k in ["MAE", "RMSE", "R2"]:
        v = scores[f"test_{k}"] * (1 if k == "R2" else -1)
        out[k] = f"{v.mean():.3f} ± {v.std():.3f}"
    return out

In [ ]:
# TODO: choose your features and target
# FEATURES = [...]
# X, y, groups = df_model[FEATURES], df_model["GPR"], df_model["PatientID"]
# evaluate(DummyRegressor(strategy="mean"), X, y, groups)

In [ ]:
# TODO: try other models (linear regression with scaling, random forest, gradient boosting, ...)

## 8. Interpretation

- Which features matter most (permutation importance, SHAP, coefficients)?
- Does adding MPC improve the prediction over aPCMs alone?
- Is the error small enough to be clinically useful? Could the model flag arcs likely to fail?

In [ ]:
# TODO